In [64]:
!pip install scikit-learn pandas

In [65]:
import pandas as pd
import re

df = pd.read_csv("mbti_1.csv")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df["clean_posts"] = df["posts"].apply(clean_text)

In [66]:
import numpy as np

def mbti_to_binary(mbti):
    return [
        1 if mbti[0] == 'I' else 0,
        1 if mbti[1] == 'N' else 0,
        1 if mbti[2] == 'T' else 0,
        1 if mbti[3] == 'J' else 0
    ]

y = np.array(df["type"].apply(mbti_to_binary).tolist())

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english'
)

X = vectorizer.fit_transform(df["clean_posts"])

In [68]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [69]:
from sklearn.linear_model import LogisticRegression

models = []

for i in range(4):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train[:, i])
    models.append(clf)

In [70]:
from sklearn.metrics import accuracy_score

for i, name in enumerate(["I/E", "N/S", "T/F", "J/P"]):
    pred = models[i].predict(X_test)
    acc = accuracy_score(y_test[:, i], pred)
    print(f"{name} Accuracy:", acc)

I/E Accuracy: 0.8409221902017291
N/S Accuracy: 0.8703170028818443
T/F Accuracy: 0.8518731988472622
J/P Accuracy: 0.7976945244956772


In [71]:
import pickle

# save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# save models
with open("mbti_models.pkl", "wb") as f:
    pickle.dump(models, f)